# Train the Structured Socratic Math Tutor (QLoRA on Qwen3-1.7B)

This notebook: gets the repo → generates the dataset → **evaluates the BASE model** → QLoRA
fine-tunes → **evaluates the TUNED model** on the same held-out set → prints the base-vs-tuned
table → interactive demo.

**Before running:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.

Getting the code into Colab (pick one):
- **GitHub (recommended):** push this project, then set `REPO_URL` below.
- **Upload:** zip the `SLM` folder, upload via the Files panel, `!unzip SLM.zip`, and `%cd SLM`.

In [ ]:
%%capture
# Unsloth pulls compatible torch/transformers/trl/peft/bitsandbytes.
!pip install unsloth
!pip install --no-deps --upgrade "trl>=0.9.6"

In [ ]:
import os

REPO_URL = ""  # e.g. "https://github.com/yourname/SLM.git"  (leave "" if you uploaded the folder)

if REPO_URL and not os.path.isdir("SLM"):
    !git clone $REPO_URL SLM
if os.path.isdir("SLM"):
    %cd SLM

assert os.path.isfile("src/generate_data.py"), "Run from the SLM project root (clone or upload it first)."

# Generate the dataset (the deliverable). Every example is quality-gated by the scorer.
!python src/generate_data.py --train 800 --val 160 --seed 7

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LEN = 1024
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-1.7B",   # fallback: "Qwen/Qwen3-1.7B"
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=None,
)

In [ ]:
# ---- Evaluate the BASE model FIRST (no training before the eval exists) ----
import sys
sys.path.insert(0, "src")
from evaluate import load_rows, run_eval, print_report

eval_rows = load_rows("eval/tutor_eval.jsonl")

FastLanguageModel.for_inference(model)
base_agg = run_eval(model, tokenizer, eval_rows, max_new_tokens=160)
print_report(base_agg, "BASE  Qwen3-1.7B  (held-out tutor_eval.jsonl)")

In [ ]:
# ---- Add LoRA adapters + build the training dataset from the chat JSONL ----
from datasets import load_dataset

model = FastLanguageModel.get_peft_model(
    model,
    r=16, lora_alpha=16, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=7,
)

def to_text(ex):
    return {"text": tokenizer.apply_chat_template(
        ex["messages"], tokenize=False, add_generation_prompt=False)}

train_ds = load_dataset("json", data_files="data/train.jsonl", split="train").map(to_text)
print(train_ds[0]["text"][:600])

In [ ]:
# ---- Train (QLoRA SFT) ----
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LEN,
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=7,
        output_dir="outputs",
        report_to="none",
    ),
)
trainer.train()

In [ ]:
# ---- Evaluate the TUNED model on the SAME held-out set + show the delta ----
FastLanguageModel.for_inference(model)
tuned_agg = run_eval(model, tokenizer, eval_rows, max_new_tokens=160)
print_report(tuned_agg, "TUNED  (held-out tutor_eval.jsonl)")

print("\n" + "=" * 72)
print(f"{'metric':<18}{'base':>8}{'tuned':>8}{'delta':>8}")
for k in ["policy_ok", "structured_exact", "diagnosis_exact", "move_legal", "leak_ok", "schema_ok"]:
    b, t = base_agg["overall"][k], tuned_agg["overall"][k]
    print(f"{k:<18}{b:>8.3f}{t:>8.3f}{t - b:>+8.3f}")

In [ ]:
# ---- Save the LoRA adapter (and optionally push to the Hugging Face Hub) ----
model.save_pretrained("outputs/tutor-lora")
tokenizer.save_pretrained("outputs/tutor-lora")
print("Saved adapter to outputs/tutor-lora")

# from huggingface_hub import login; login()  # paste your HF token
# model.push_to_hub("your-username/qwen3-1.7b-socratic-tutor", token=True)
# tokenizer.push_to_hub("your-username/qwen3-1.7b-socratic-tutor", token=True)

In [ ]:
# ---- Interactive demo: watch the tuned model do what the base couldn't ----
import json, torch
from tutor.policy import SYSTEM_PROMPT

def tutor_move(problem, student, mdl=model, tok=tokenizer):
    user = f"PROBLEM: {problem}\nSTUDENT: {student}"
    text = tok.apply_chat_template(
        [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(text, return_tensors="pt").to(mdl.device)
    with torch.no_grad():
        out = mdl.generate(**inputs, max_new_tokens=160, do_sample=False, pad_token_id=tok.eos_token_id)
    reply = tok.decode(out[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True).strip()
    try:
        return json.dumps(json.loads(reply), indent=2)
    except Exception:
        return reply

print(tutor_move("What is 20% of 50?", "I got 1000"))                 # -> wrong_operation, give_hint, no leak
print(tutor_move("Solve for x: 3x + 2 = 20", "just tell me the answer"))  # -> redirect_no_answer
print(tutor_move("What is 6 * 9?", "is it 54?"))                       # -> correct_answer, affirm, reveals